In [ ]:
import os
import pandas as pd

base_path = "../../../data"
output_path = "output"

os.makedirs(output_path, exist_ok=True)

# mappa sensori → colonna finale
column_map = {
    "barometro": "barometer",
    "termometro": "temperature",
    "vento_scalare": "wind_speed",
    "vento_raffica": "wind_gust",
    "pluviometro": "rain_rate"
}

for location in os.listdir(base_path):
    loc_path = os.path.join(base_path, location)
    
    if not os.path.isdir(loc_path):
        continue

    print(f"Processing {location}...")

    merged_df = None

    for sensor_file in os.listdir(loc_path):
        sensor_name = os.path.splitext(sensor_file)[0]  # "barometro", "termometro", ...
        
        if sensor_name not in column_map:
            print(f"Skipped file {sensor_file}")
            continue
        
        sensor_path = os.path.join(loc_path, sensor_file)
        
        df = pd.read_csv(sensor_path, sep=';')
        df.columns = ["Date", "Time", "Value", "Unit"]
        
        # corregge virgole decimali
        df["Value"] = df["Value"].astype(str).str.replace(",", ".").astype(float)
        
        # crea timestamp completo
        df["datetime"] = df["Date"] + " " + df["Time"]
        
        df = df[["datetime", "Value"]].rename(columns={"Value": column_map[sensor_name]})
        
        if merged_df is None:
            merged_df = df
        else:
            merged_df = pd.merge(merged_df, df, on="datetime", how="outer")

    # split per avere Date + Time come richiesto
    merged_df[["Date", "Time"]] = merged_df["datetime"].str.split(" ", expand=True)
    merged_df = merged_df.drop(columns=["datetime"])

    # ordina e imposta ordine colonne
    ordered_cols = ["Date", "Time"] + list(column_map.values())
    merged_df = merged_df[ordered_cols]

    # salva
    year = merged_df["Date"].iloc[0].split("-")[2]  # estrae 2025 da "01-01-2025"
    output_file = os.path.join(output_path, f"{year}_{location}.csv")
    merged_df.to_csv(output_file, sep=";", index=False)

    print(f"Saved → {output_file}")